# KG1 V71 MEGA FINAL — Colab Pro H100 / A100 HighRAM

Integrates findings from 13+ agents:
- D1 metric fixes (math.isclose rel_tol=1e-2 abs_tol=1e-5, enable_thinking=True, BOXED_INSTRUCTION byte-for-byte)
- V1 bit_manipulation_pairs (58% empirical)
- V2 cryptarithm_47combo (17% empirical)
- V5 max_min_logprob loss (warmup CE 100 steps, grad_clip=1.0)
- T1 vLLM config (max_length=8192, mamba_ssm_cache_dtype=float32, attn=eager)
- T2 LoRA baseline r=32 alpha=32 all-linear
- T3 category-aware prompts (spacing ONLY cipher/cryptarithm)
- T5 neurosymbolic template
- T6 metric exploits (no units/commas/LaTeX/nested inside boxed)
- V6 datasets integration
- V7 risk dossier pre-flight checks

Target: 0.84 baseline -> 0.87+ (99% rule gate before submit)

## Cell 1 — Setup dependencies

In [ ]:
# Cell 1: install deps for NemotronH LoRA SFT (resilient install)
# FIX 2026-04-21: version pins were too tight causing pip resolver to fail.
# Now installs each package individually with continue-on-fail, then retries batch.
import subprocess, sys

def _pip_one(pkg, extra_args=None):
    """Install single package. Returns True on success, False on failure."""
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', pkg]
    if extra_args:
        cmd.extend(extra_args)
    print(f'pip install {pkg} ...', end=' ', flush=True)
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    except subprocess.TimeoutExpired:
        print('TIMEOUT')
        return False
    if r.returncode == 0:
        print('OK')
        return True
    else:
        print('FAIL')
        print('  stderr tail:', r.stderr[-500:])
        return False

# Upgrade pip first
_pip_one('pip', extra_args=['--upgrade'])

# Relaxed version pins (just minimum, no upper bound — let pip resolve compat)
CORE_PKGS = [
    'transformers>=4.55',
    'peft>=0.13',
    'trl>=0.25',
    'accelerate>=0.34',
    'bitsandbytes>=0.44',
    'datasets>=2.20',
    'safetensors>=0.4.5',
    'sentencepiece',
    'einops',
    'huggingface_hub>=0.25',
]

failed = []
for pkg in CORE_PKGS:
    if not _pip_one(pkg):
        failed.append(pkg)

# Retry any failures with --upgrade flag in batch
if failed:
    print(f'\nRetrying {len(failed)} failed packages with --upgrade...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + failed,
        capture_output=True, text=True, timeout=900,
    )
    if r.returncode != 0:
        print('Batch retry also failed — attempting individual --upgrade:')
        for p in failed:
            _pip_one(p, extra_args=['--upgrade'])

# mamba-ssm + causal-conv1d OPTIONAL — NemotronH falls back to slow path if missing
print('\n--- Installing mamba-ssm + causal-conv1d (OPTIONAL, slow path fallback) ---')
try:
    import mamba_ssm  # noqa: F401
    print('mamba-ssm already installed')
except ImportError:
    if not _pip_one('mamba-ssm', extra_args=['--no-build-isolation']):
        print('WARNING: mamba-ssm failed — slow path fallback (training may be 2x slower)')

try:
    import causal_conv1d  # noqa: F401
    print('causal-conv1d already installed')
except ImportError:
    if not _pip_one('causal-conv1d>=1.4', extra_args=['--no-build-isolation']):
        print('WARNING: causal-conv1d failed — slow path fallback')

# Final verification
import torch
import transformers, peft, trl
print('\n=== Installed versions ===')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')
print(f'trl:          {trl.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device = torch.cuda.get_device_properties(0)
    vram_gb = device.total_memory / 1024**3
    print(f'GPU: {device.name} ({vram_gb:.1f} GB VRAM)')
    if vram_gb < 38:
        print(f'WARNING: VRAM {vram_gb:.1f}GB < 38GB recommended — may OOM on 30B model')
        print('  Switch to A100 HighRAM (40GB) or H100 (80GB) for safe training')
else:
    raise RuntimeError('CUDA not available — Runtime -> Change runtime type -> A100 or H100')

print('\nCell 1 DONE')


## Cell 2 — Clone KG1 worktree (git shallow) or upload via GDrive zip

In [ ]:
# Cell 2: bring KG1 source into /content/kg1
import os, subprocess, shutil
from pathlib import Path

KG1_DIR = Path('/content/kg1')
REPO_SSH = 'https://github.com/FELIPEACASTRO/KG1-NVIDIA.git'
BRANCH = os.environ.get('KG1_BRANCH', 'claude/competent-shamir')

if KG1_DIR.exists() and (KG1_DIR / 'src').exists():
    print('KG1 already present at', KG1_DIR)
else:
    # Try GDrive zip fallback first if available (offline / no git access)
    gdrive_zip = Path('/content/drive/MyDrive/kg1_src.zip')
    if gdrive_zip.exists():
        print('Unzipping KG1 from GDrive:', gdrive_zip)
        shutil.unpack_archive(str(gdrive_zip), str(KG1_DIR))
    else:
        print('Cloning from GitHub:', REPO_SSH, 'branch=', BRANCH)
        subprocess.check_call([
            'git', 'clone', '--depth', '1', '--branch', BRANCH,
            REPO_SSH, str(KG1_DIR),
        ])

# Assertions — crucial source files must exist
REQUIRED = [
    'src/reasoners/bit_manipulation_pairs.py',
    'src/reasoners/cryptarithm_47combo.py',
    'src/reasoners/neurosymbolic_template.py',
    'src/losses/max_min_logprob.py',
    'src/prompts/build_prompt.py',
    'scripts/local_score.py',
    'scripts/kg1_submission_gate.py',
]
missing = [r for r in REQUIRED if not (KG1_DIR / r).exists()]
assert not missing, f'Missing source files: {missing}'
print('All required source files present.')

# Put KG1 on sys.path
import sys
if str(KG1_DIR) not in sys.path:
    sys.path.insert(0, str(KG1_DIR))
print('sys.path[0] =', sys.path[0])


## Cell 3 — GDrive + HF token + Kaggle creds

In [ ]:
# Cell 3: mount GDrive, load HF_KEY (Colab secret) + KAGGLE creds
import os, json
from pathlib import Path

try:
    from google.colab import drive, userdata  # type: ignore
    drive.mount('/content/drive')
    # IMPORTANT: user memory says secret is HF_KEY (not HF_TOKEN)
    try:
        hf_key = userdata.get('HF_KEY')
    except Exception:
        hf_key = None
    if not hf_key:
        hf_key = os.environ.get('HF_KEY') or os.environ.get('HF_TOKEN')
    assert hf_key, 'HF_KEY not found — add it to Colab secrets'
    os.environ['HF_TOKEN'] = hf_key
    os.environ['HF_KEY'] = hf_key

    try:
        kuser = userdata.get('KAGGLE_USERNAME')
        kkey = userdata.get('KAGGLE_KEY')
    except Exception:
        kuser = os.environ.get('KAGGLE_USERNAME')
        kkey = os.environ.get('KAGGLE_KEY')
    if kuser and kkey:
        os.environ['KAGGLE_USERNAME'] = kuser
        os.environ['KAGGLE_KEY'] = kkey
        kpath = Path.home() / '.kaggle' / 'kaggle.json'
        kpath.parent.mkdir(parents=True, exist_ok=True)
        kpath.write_text(json.dumps({'username': kuser, 'key': kkey}))
        kpath.chmod(0o600)
        print('Kaggle creds installed for user', kuser)
    else:
        print('WARNING: Kaggle creds not set (needed only for Cell 15)')
except ImportError:
    # Not Colab — fall back to env vars
    print('Not running in Colab — relying on env vars HF_TOKEN/HF_KEY/KAGGLE_*')
    assert os.environ.get('HF_TOKEN') or os.environ.get('HF_KEY'), 'HF token required'

from huggingface_hub import HfFolder
HfFolder.save_token(os.environ.get('HF_TOKEN') or os.environ.get('HF_KEY'))
print('HF token saved.')


## Cell 4 — V71 Config (all agent findings)

In [ ]:
# Cell 4: V71 config — every flag traced to an agent finding
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json, os

@dataclass
class V71Config:
    # --- Model (T1 vLLM config + T2 LoRA baseline) ---
    base_model: str = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
    max_length: int = 8192                 # T1: 8192 not 4096
    attn_implementation: str = 'eager'      # T1: mamba hybrid requires eager
    mamba_ssm_cache_dtype: str = 'float32'  # T1: numerical stability
    tie_word_embeddings: bool = False       # V7: must be False (V18 incident)

    # --- LoRA (T2 baseline r=32 alpha=32 all-linear) ---
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: str = 'all-linear'
    use_dora: bool = False                   # T2 V71b ablation toggle
    use_rank_pattern: bool = False           # T2 V71c ablation toggle

    # --- Training schedule ---
    epochs: int = 1
    per_device_batch: int = 1
    grad_accum: int = 16                     # effective batch 16 on H100 80GB
    learning_rate: float = 2e-4
    lr_scheduler: str = 'linear'
    warmup_ratio: float = 0.03
    grad_clip: float = 1.0                   # V5: mandatory for max-min
    optimizer: str = 'paged_adamw_8bit'
    bf16: bool = True
    gradient_checkpointing: bool = False     # V16.2: NemotronH incompatible

    # --- Loss (V5 max-min logprob with CE warmup) ---
    loss_type: str = 'max_min_warmup_ce'    # 'ce' | 'max_min' | 'max_min_warmup_ce'
    max_min_warmup_steps: int = 100          # V5: CE warmup then switch

    # --- Data (V6 + V71 corpus) ---
    data_sources: list = field(default_factory=lambda: [
        'data/v71/weak_prompts.jsonl',
        'data/external/jasonkung98_NVIDIA-Nemotron-Model-Reasoning-Challenge/train.csv',
        'data/external/nvidia_Nemotron-RL-ReasoningGym-v1/',
        'data/external/nvidia_Puzzle-KD-Nemotron-Post-Training-Dataset-v2/',
    ])
    augment_bit_manipulation: bool = True   # V1 programmatic CoT
    augment_cryptarithm: bool = True        # V2 programmatic CoT

    # --- Prompt construction (T3 category-aware + D1 byte-for-byte) ---
    enable_thinking: bool = True             # D1: apply_chat_template arg
    use_structured: bool = True              # T3 QuaSAR
    use_category_hints: bool = True          # T3 cipher/cryptarithm only
    use_boxed_strict: bool = True            # T6 defensive
    use_self_correct: bool = True            # T6 Exploit 15 (training only)

    # --- Gate thresholds (V7 risk dossier) ---
    smoke_test_steps: int = 2
    smoke_abort_loss: float = 50.0           # V7: abort if loss > 50 at step 2
    eval_holdout_size: int = 600             # local eval on 600 rows
    local_score_floor: float = 0.84          # 99% rule gate floor
    target_score: float = 0.87               # Top1 target

    # --- Output ---
    run_tag: str = 'v71_mega'
    output_dir: str = '/content/kg1_out/v71_mega'
    gdrive_checkpoint: str = '/content/drive/MyDrive/kg1_checkpoints/v71_mega'
    hf_upload_repo: str = 'felipesp1983/kg1-nemotron-lora-v71-mega'

CFG = V71Config()
# Sanity checks on config invariants
assert CFG.grad_clip > 0, 'grad_clip must be > 0 for max-min stability'
assert CFG.max_length <= 16384, 'max_length above sane range'
assert CFG.attn_implementation == 'eager', 'NemotronH hybrid requires eager'
assert CFG.tie_word_embeddings is False, 'tie_word_embeddings must be False (V18 incident)'
assert CFG.loss_type in ('ce', 'max_min', 'max_min_warmup_ce')

Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
with open(Path(CFG.output_dir) / 'config.json', 'w') as f:
    json.dump(asdict(CFG), f, indent=2)
print('V71 config written to', Path(CFG.output_dir) / 'config.json')
print(json.dumps(asdict(CFG), indent=2)[:1200])


## Cell 5 — Pre-flight checks (V7 dossier)

In [ ]:
# Cell 5: pre-flight — VRAM, tokenizer, data integrity, adapter config
import torch, importlib, os, sys
from pathlib import Path

# 5.1 VRAM check — need >=40GB for NemotronH 30B BF16 + LoRA
device_props = torch.cuda.get_device_properties(0)
total_vram_gb = device_props.total_memory / (1024 ** 3)
print(f'GPU: {device_props.name}, VRAM: {total_vram_gb:.1f} GB')
assert total_vram_gb >= 38, f'Need >=38GB VRAM, got {total_vram_gb:.1f}'

# 5.2 Smoke-import all KG1 modules
for mod_name in [
    'src.reasoners.bit_manipulation_pairs',
    'src.reasoners.cryptarithm_47combo',
    'src.reasoners.neurosymbolic_template',
    'src.losses.max_min_logprob',
    'src.prompts.build_prompt',
]:
    mod = importlib.import_module(mod_name)
    print('imported', mod_name, 'from', getattr(mod, '__file__', '?'))

# 5.3 Tokenizer load + assertion
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG.base_model, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
assert tok.pad_token is not None, 'tokenizer has no pad token'
print('tokenizer loaded. vocab_size=', tok.vocab_size, 'pad=', tok.pad_token)

# 5.4 enable_thinking support check (D1)
try:
    sample_msg = [{'role': 'user', 'content': 'test'}]
    _ = tok.apply_chat_template(sample_msg, enable_thinking=True, tokenize=False)
    print('enable_thinking=True supported by tokenizer chat template')
except TypeError as e:
    print('WARNING: enable_thinking kwarg not accepted by this tokenizer —', e)

# 5.5 Self-test imported reasoners (V1, V2, T5)
from src.reasoners.bit_manipulation_pairs import generate_cot as gen_bit
pred, cot = gen_bit([('00000000', '10101010'), ('11111111', '01010101')], '10101010')
assert pred is not None, 'bit_manipulation_pairs failed on self-test'
print('bit_manipulation self-test OK, pred=', pred)

from src.prompts.build_prompt import build_prompt_v71
p = build_prompt_v71('Decrypt: HELLO -> Ifmmp. What is World?', 'cipher')
assert '\\boxed' in p, 'BOXED_INSTRUCTION missing from prompt'
print('build_prompt_v71 OK, length=', len(p))

print('\nAll pre-flight checks PASSED.')


## Cell 6 — Load training data + programmatic CoT augmentation

In [ ]:
# Cell 6: load training data + apply programmatic CoT augmentation for weak cats
import json, pandas as pd
from pathlib import Path
from src.reasoners.bit_manipulation_pairs import generate_cot as gen_bit_cot
from src.reasoners.cryptarithm_47combo import generate_cot as gen_crypt_cot
from src.prompts.build_prompt import build_prompt_v71, detect_category

# Training data file paths (prefer GDrive for speed)
TRAIN_CSV_CANDIDATES = [
    Path('/content/drive/MyDrive/kg1_data/train.csv'),
    Path('/content/kg1/data/kaggle/unzipped/train.csv'),
    Path('/content/kg1/data/external/jasonkung98_NVIDIA-Nemotron-Model-Reasoning-Challenge/train.csv'),
]
train_csv = next((p for p in TRAIN_CSV_CANDIDATES if p.exists()), None)
assert train_csv is not None, f'No train.csv found — tried {TRAIN_CSV_CANDIDATES}'
print('Loading', train_csv)
df = pd.read_csv(train_csv)
print('rows=', len(df), 'cols=', list(df.columns))

# Normalize expected columns
assert 'prompt' in df.columns, 'expected column "prompt" in train.csv'
ans_col = 'answer' if 'answer' in df.columns else ('target' if 'target' in df.columns else None)
assert ans_col is not None, 'expected "answer" or "target" column'
df = df.rename(columns={ans_col: 'answer'})

# Attach category (from col or detect)
if 'category' not in df.columns:
    df['category'] = df['prompt'].map(detect_category)

# --- Programmatic CoT augmentation (V1 + V2) ---
# Extract examples/query from prompt text. Use regex for simple common formats; rows where
# extraction fails are just skipped (kept with empty programmatic_cot).
import re
BIT_EX_RE = re.compile(r'([01]{8})\s*->\s*([01]{8})')
BIT_Q_RE = re.compile(r'[Ww]hat is\s+([01]{8})')
CRYPT_EX_RE = re.compile(r'([^\s=]{5})\s*=\s*(\S+)')
CRYPT_Q_RE = re.compile(r'[Qq]uery\s*:\s*(\S+)')

def augment_row(row):
    prompt_text, cat = str(row['prompt']), str(row.get('category', ''))
    if CFG.augment_bit_manipulation and cat == 'bit_manipulation':
        exs = BIT_EX_RE.findall(prompt_text)
        qm = BIT_Q_RE.search(prompt_text)
        if exs and qm:
            pred, cot = gen_bit_cot(exs, qm.group(1))
            if pred is not None:
                return cot
    if CFG.augment_cryptarithm and cat in ('cryptarithm_deduce', 'cryptarithm_guess'):
        exs = CRYPT_EX_RE.findall(prompt_text)
        qm = CRYPT_Q_RE.search(prompt_text)
        if exs and qm:
            pred, cot = gen_crypt_cot(exs, qm.group(1))
            if pred is not None:
                return cot
    return ''

df['programmatic_cot'] = df.apply(augment_row, axis=1)
aug_count = int((df['programmatic_cot'] != '').sum())
print(f'Programmatic CoT augmented rows: {aug_count}/{len(df)}')

# Build final training records: (prompt with category hints, completion)
def build_record(row):
    cat = str(row.get('category', ''))
    user = build_prompt_v71(
        str(row['prompt']),
        category=cat,
        use_structured=CFG.use_structured,
        use_category_hints=CFG.use_category_hints,
        use_boxed_strict=CFG.use_boxed_strict,
        use_self_correct=CFG.use_self_correct,
    )
    gt = str(row['answer'])
    cot = row.get('programmatic_cot', '') or ''
    assistant = (cot + '\n' if cot else '') + f'Final answer: {gt}\n\\boxed{{{gt}}}'
    return {'user': user, 'assistant': assistant, 'category': cat}

records = df.apply(build_record, axis=1).tolist()
print(f'Built {len(records)} training records.')

# Split train/eval
import random
random.seed(42)
indices = list(range(len(records)))
random.shuffle(indices)
eval_n = min(CFG.eval_holdout_size, max(50, len(records) // 20))
eval_idx = set(indices[:eval_n])
train_records = [records[i] for i in indices if i not in eval_idx]
eval_records = [records[i] for i in indices if i in eval_idx]
print(f'train={len(train_records)}  eval={len(eval_records)}')

# Persist to disk for Trainer
out_path = Path(CFG.output_dir) / 'train.jsonl'
with open(out_path, 'w') as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
eval_path = Path(CFG.output_dir) / 'eval.jsonl'
with open(eval_path, 'w') as f:
    for r in eval_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('wrote', out_path, eval_path)


## Cell 7 — Build model + LoRA config

In [ ]:
# Cell 7: load NemotronH BF16 + attach PEFT LoRA per V71 config
import torch
from transformers import AutoModelForCausalLM, AutoConfig
from peft import LoraConfig, get_peft_model, TaskType

# Load config first so we can enforce tie_word_embeddings and mamba dtype
model_cfg = AutoConfig.from_pretrained(CFG.base_model, trust_remote_code=True)
# Enforce V7 invariant
if getattr(model_cfg, 'tie_word_embeddings', False):
    print('WARN: base config has tie_word_embeddings=True — forcing to False (V18 incident)')
setattr(model_cfg, 'tie_word_embeddings', CFG.tie_word_embeddings)
# T1 numerical stability for SSM cache
if hasattr(model_cfg, 'mamba_ssm_cache_dtype'):
    setattr(model_cfg, 'mamba_ssm_cache_dtype', CFG.mamba_ssm_cache_dtype)

model = AutoModelForCausalLM.from_pretrained(
    CFG.base_model,
    config=model_cfg,
    torch_dtype=torch.bfloat16 if CFG.bf16 else torch.float16,
    device_map='auto',
    attn_implementation=CFG.attn_implementation,  # 'eager'
    trust_remote_code=True,
)
# Safety assertion (post-load)
assert not getattr(model.config, 'tie_word_embeddings', False), 'tie_word_embeddings leaked to True after load'
print('Base model loaded. dtype=', next(model.parameters()).dtype)

# Attach LoRA
rank_pattern = None
if CFG.use_rank_pattern:
    # V71c ablation — heterogeneous ranks (up-proj richer, down-proj leaner)
    rank_pattern = {
        r'.*up_proj': 64,
        r'.*gate_proj': 48,
        r'.*down_proj': 16,
    }

lora_kwargs = dict(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=CFG.lora_target_modules,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    use_dora=CFG.use_dora,
)
if rank_pattern is not None:
    lora_kwargs['rank_pattern'] = rank_pattern

peft_cfg = LoraConfig(**lora_kwargs)
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

if CFG.gradient_checkpointing:
    # V16.2 flag — only enable if future NemotronH version supports it
    try:
        model.gradient_checkpointing_enable()
    except Exception as e:
        print('gradient_checkpointing_enable failed (expected for NemotronH):', e)

# Final invariants
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {n_trainable:,} / Total: {n_total:,}  ({100 * n_trainable / n_total:.3f}%)')
assert 0 < n_trainable < n_total, 'LoRA attach failed'


## Cell 8 — SMOKE TEST (2 steps)

In [ ]:
# Cell 8: smoke test 2 steps — assert loss < abort threshold, no NaN, monotonic
import math, torch, json
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from src.losses.max_min_logprob import max_min_logprob_loss

class JsonlChatDataset(Dataset):
    def __init__(self, path, tokenizer, max_len):
        self.rows = [json.loads(l) for l in open(path)]
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        messages = [
            {'role': 'user', 'content': r['user']},
            {'role': 'assistant', 'content': r['assistant']},
        ]
        try:
            text = self.tok.apply_chat_template(
                messages, tokenize=False, enable_thinking=CFG.enable_thinking,
            )
        except TypeError:
            text = self.tok.apply_chat_template(messages, tokenize=False)
        enc = self.tok(text, truncation=True, max_length=self.max_len, return_tensors='pt')
        ids = enc['input_ids'][0]
        # labels = ids with prompt portion masked out. Approximation: mask all until last 'assistant' header marker.
        labels = ids.clone()
        try:
            user_text = self.tok.apply_chat_template([messages[0]], tokenize=False, enable_thinking=CFG.enable_thinking)
        except TypeError:
            user_text = self.tok.apply_chat_template([messages[0]], tokenize=False)
        user_ids = self.tok(user_text, return_tensors='pt')['input_ids'][0]
        k = min(len(user_ids), len(labels))
        labels[:k] = -100
        return {'input_ids': ids, 'labels': labels, 'attention_mask': enc['attention_mask'][0]}

def collate(batch, pad_id):
    max_l = max(x['input_ids'].size(0) for x in batch)
    def pad(t, v):
        return torch.nn.functional.pad(t, (0, max_l - t.size(0)), value=v)
    return {
        'input_ids': torch.stack([pad(x['input_ids'], pad_id) for x in batch]),
        'labels': torch.stack([pad(x['labels'], -100) for x in batch]),
        'attention_mask': torch.stack([pad(x['attention_mask'], 0) for x in batch]),
    }

train_path = Path(CFG.output_dir) / 'train.jsonl'
ds = JsonlChatDataset(train_path, tok, CFG.max_length)
dl = DataLoader(
    ds, batch_size=CFG.per_device_batch, shuffle=True,
    collate_fn=lambda b: collate(b, tok.pad_token_id),
)

model.train()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)  # smaller LR for smoke
losses = []
for step, batch in enumerate(dl):
    batch = {k: v.to(model.device) for k, v in batch.items()}
    out = model(**{k: v for k, v in batch.items() if k != 'labels'})
    # Smoke uses plain CE — switching loss function is validated in Cell 9.
    loss = torch.nn.functional.cross_entropy(
        out.logits.view(-1, out.logits.size(-1)), batch['labels'].view(-1), ignore_index=-100,
    )
    losses.append(loss.item())
    assert not math.isnan(loss.item()), f'NaN loss at step {step}'
    assert not math.isinf(loss.item()), f'Inf loss at step {step}'
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], CFG.grad_clip)
    opt.step()
    opt.zero_grad(set_to_none=True)
    print(f'smoke step {step} loss={loss.item():.4f}')
    if step + 1 >= CFG.smoke_test_steps:
        break

assert len(losses) == CFG.smoke_test_steps, f'expected {CFG.smoke_test_steps} steps, got {len(losses)}'
# V7 abort gate: final loss must be below abort threshold
assert losses[-1] < CFG.smoke_abort_loss, (
    f'ABORT: smoke loss {losses[-1]:.2f} > abort threshold {CFG.smoke_abort_loss}'
)
# We do NOT require monotonic decrease over 2 steps (noise). We only require no explosion.
print('Smoke test PASSED.')


## Cell 9 — Full training with MaxMinLogProbTrainer

In [ ]:
# Cell 9: full 1-epoch training with custom SFTTrainer overriding compute_loss
import torch, os
from pathlib import Path
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from src.losses.max_min_logprob import max_min_logprob_loss

train_path = str(Path(CFG.output_dir) / 'train.jsonl')
eval_path = str(Path(CFG.output_dir) / 'eval.jsonl')

ds_train = load_dataset('json', data_files=train_path, split='train')
ds_eval = load_dataset('json', data_files=eval_path, split='train')

def format_example(ex):
    messages = [
        {'role': 'user', 'content': ex['user']},
        {'role': 'assistant', 'content': ex['assistant']},
    ]
    try:
        text = tok.apply_chat_template(
            messages, tokenize=False, enable_thinking=CFG.enable_thinking,
        )
    except TypeError:
        text = tok.apply_chat_template(messages, tokenize=False)
    return {'text': text}

ds_train = ds_train.map(format_example, remove_columns=ds_train.column_names)
ds_eval = ds_eval.map(format_example, remove_columns=ds_eval.column_names)

sft_args = SFTConfig(
    output_dir=CFG.output_dir,
    per_device_train_batch_size=CFG.per_device_batch,
    per_device_eval_batch_size=CFG.per_device_batch,
    gradient_accumulation_steps=CFG.grad_accum,
    num_train_epochs=CFG.epochs,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type=CFG.lr_scheduler,
    warmup_ratio=CFG.warmup_ratio,
    max_grad_norm=CFG.grad_clip,
    bf16=CFG.bf16,
    logging_steps=10,
    save_steps=200,
    eval_strategy='no',
    save_total_limit=2,
    optim=CFG.optimizer,
    max_seq_length=CFG.max_length,
    packing=False,
    report_to=[],
    gradient_checkpointing=CFG.gradient_checkpointing,
    dataset_text_field='text',
)

class MaxMinSFTTrainer(SFTTrainer):
    """Trainer that warms up on CE then switches to max-min logprob loss (V5)."""
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**{k: v for k, v in inputs.items() if k != 'labels'})
        logits = outputs.logits
        step = int(self.state.global_step)
        if CFG.loss_type == 'ce':
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100,
            )
        elif CFG.loss_type == 'max_min':
            loss = max_min_logprob_loss(logits, labels)
        else:  # 'max_min_warmup_ce'
            if step < CFG.max_min_warmup_steps:
                loss = torch.nn.functional.cross_entropy(
                    logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100,
                )
            else:
                loss = max_min_logprob_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = MaxMinSFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tok,
)
trainer.train()
print('Training complete. Saving adapter...')
trainer.save_model(CFG.output_dir)
tok.save_pretrained(CFG.output_dir)
print('Saved to', CFG.output_dir)


## Cell 10 — Save adapter + upload to HF + checkpoint to GDrive

In [ ]:
# Cell 10: save adapter, checkpoint to GDrive (colab-agent skill pattern), push to HF Hub
import shutil, datetime, os
from pathlib import Path
from huggingface_hub import HfApi, upload_folder

out_dir = Path(CFG.output_dir)
# Safety: adapter files present?
required_files = ['adapter_config.json', 'adapter_model.safetensors']
present = [f for f in required_files if (out_dir / f).exists()]
assert len(present) == len(required_files), f'Missing adapter files: {set(required_files) - set(present)}'

# GDrive checkpoint with timestamp (colab-checkpoint pattern)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
gdrive_dest = Path(CFG.gdrive_checkpoint) / f'{CFG.run_tag}_{ts}'
try:
    gdrive_dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(out_dir, gdrive_dest, dirs_exist_ok=True)
    print('Checkpoint saved to GDrive:', gdrive_dest)
except Exception as e:
    print('WARN: GDrive save failed (non-fatal):', e)

# HF upload
api = HfApi(token=os.environ.get('HF_TOKEN') or os.environ.get('HF_KEY'))
try:
    api.create_repo(CFG.hf_upload_repo, private=True, exist_ok=True)
    upload_folder(
        repo_id=CFG.hf_upload_repo,
        folder_path=str(out_dir),
        allow_patterns=['adapter_*', 'tokenizer*', 'special_tokens*', 'config.json'],
        token=os.environ.get('HF_TOKEN') or os.environ.get('HF_KEY'),
    )
    print('Uploaded adapter to HF:', CFG.hf_upload_repo)
except Exception as e:
    print('WARN: HF upload failed (non-fatal):', e)


## Cell 11 — Local eval on 600-row holdout (CORRECTED metric)

In [ ]:
# Cell 11: run local_score.py in subprocess with CORRECTED metric (math.isclose rel_tol=1e-2 abs_tol=1e-5)
import subprocess, json, os, sys
from pathlib import Path

local_score = Path('/content/kg1/scripts/local_score.py')
assert local_score.exists(), f'local_score.py missing at {local_score}'
eval_csv = Path(CFG.output_dir) / 'local_eval.csv'
adapter_arg = str(Path(CFG.output_dir))

cmd = [
    sys.executable, str(local_score),
    '--adapter', adapter_arg,
    '--n-samples', str(CFG.eval_holdout_size),
    '--output-csv', str(eval_csv),
]
print('Running:', ' '.join(cmd))
try:
    res = subprocess.run(cmd, cwd='/content/kg1', check=False, capture_output=True, text=True, timeout=3600)
    print('STDOUT (tail):', res.stdout[-2000:])
    if res.returncode != 0:
        print('STDERR (tail):', res.stderr[-2000:])
except subprocess.TimeoutExpired:
    print('local_score timed out after 1h — using fallback manual eval')

# Parse score — local_score.py emits lines like 'Overall score: 0.84xx'
local_score_val = None
import re
OUT = res.stdout if 'res' in dir() else ''
m = re.search(r'(?:overall\s+score|score)[:\s]+([0-9.]+)', OUT, re.IGNORECASE)
if m:
    local_score_val = float(m.group(1))
print('Parsed local score =', local_score_val)

with open(Path(CFG.output_dir) / 'local_score.json', 'w') as f:
    json.dump({'local_score': local_score_val, 'n_samples': CFG.eval_holdout_size}, f)


## Cell 12 — Gate decision (99% rule)

In [ ]:
# Cell 12: GO / NO-GO gate based on local score (99% rule)
import json
from pathlib import Path

d = json.loads((Path(CFG.output_dir) / 'local_score.json').read_text())
score = d.get('local_score')

GO = False
if score is None:
    msg = 'NO-GO: could not parse local_score — skipping submission per 99% rule'
elif score < CFG.local_score_floor:
    msg = f'NO-GO: local_score {score:.4f} < floor {CFG.local_score_floor} — would regress baseline'
elif score < CFG.target_score - 0.01:
    msg = f'MARGINAL: local_score {score:.4f} within 1pp of target {CFG.target_score} — submit with caution'
    GO = True
else:
    msg = f'GO: local_score {score:.4f} >= target {CFG.target_score}'
    GO = True

print(msg)
with open(Path(CFG.output_dir) / 'gate_decision.json', 'w') as f:
    json.dump({'go': GO, 'msg': msg, 'score': score}, f)

# Hard enforcement — downstream cells should check gate_decision.json before submitting
if not GO:
    print('DO NOT run Cells 13-15 — rerun training with different config or roll back to V70.')


## Cell 13 — Prepare Kaggle submission ZIP

In [ ]:
# Cell 13: build submission ZIP (2 files at root), respecting reference_submission_format.md
import zipfile, json, os
from pathlib import Path

# Gate check
gate = json.loads((Path(CFG.output_dir) / 'gate_decision.json').read_text())
assert gate.get('go'), f'Gate is NO-GO: {gate.get("msg")}'

out_dir = Path(CFG.output_dir)
zip_path = out_dir / 'submission.zip'

# Files required at ZIP ROOT per competition format:
# - adapter_config.json
# - adapter_model.safetensors
ROOT_FILES = ['adapter_config.json', 'adapter_model.safetensors']
for fn in ROOT_FILES:
    assert (out_dir / fn).exists(), f'missing {fn} at {out_dir}'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ROOT_FILES:
        zf.write(out_dir / fn, arcname=fn)

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'submission.zip built: {zip_path}  ({size_mb:.2f} MB)')
# V18 reminder — zip must NOT contain lm_head.* weights
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    assert len(names) == 2, f'expected 2 files at root, got {names}'
    # lm_head pattern check left to kg1_submission_gate in Cell 14
print('Contents:', names)


## Cell 14 — kg1_submission_gate

In [ ]:
# Cell 14: run kg1_submission_gate.py to double-check zip integrity
import subprocess, sys
from pathlib import Path

gate_script = Path('/content/kg1/scripts/kg1_submission_gate.py')
zip_path = Path(CFG.output_dir) / 'submission.zip'
cmd = [sys.executable, str(gate_script), '--zip', str(zip_path)]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', res.stdout[-1500:])
print('STDERR:', res.stderr[-500:])
assert res.returncode == 0, f'kg1_submission_gate rejected zip (rc={res.returncode})'
print('Submission gate PASSED.')


## Cell 15 — Submit to Kaggle (respects 5/day hard limit)

In [ ]:
# Cell 15: submit via Kaggle API. Check remaining slots first (5/day hard limit).
import os, json, subprocess, sys, datetime
from pathlib import Path

assert os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'), 'Kaggle creds missing'

# Use submit_kaggle.py wrapper if available (honors 5/day limit internally)
submit_script = Path('/content/kg1/scripts/submit_kaggle.py')
zip_path = Path(CFG.output_dir) / 'submission.zip'

msg = f'V71_MEGA_FINAL {datetime.datetime.now().strftime("%Y-%m-%d %H:%M BRT")}'

if submit_script.exists():
    cmd = [sys.executable, str(submit_script), '--zip', str(zip_path), '--message', msg]
else:
    # Fallback: direct kaggle CLI
    cmd = [
        'kaggle', 'competitions', 'submit',
        '-c', 'nvidia-open-model-kaggle-grandmasters-panel',
        '-f', str(zip_path), '-m', msg,
    ]

print('Submit cmd:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', res.stdout)
print('STDERR:', res.stderr)

# Persist submission metadata
with open(Path(CFG.output_dir) / 'kaggle_submit.json', 'w') as f:
    json.dump({
        'msg': msg,
        'returncode': res.returncode,
        'stdout_tail': res.stdout[-1000:],
        'stderr_tail': res.stderr[-500:],
    }, f)
print('Submission dispatched. Monitor at https://www.kaggle.com/competitions (Submissions tab)')


## Cell 16 — Decision tree based on score

In [ ]:
# Cell 16: interpret local/Kaggle score and decide next action (Stage 2/3/abort)
import json
from pathlib import Path

gate = json.loads((Path(CFG.output_dir) / 'gate_decision.json').read_text())
score = gate.get('score') or 0.0

DECISION_TREE = [
    (0.87, 'TOP1_CANDIDATE',    'Score >= 0.87 — Stage 2 (LoRA Soup DARE-TIES) + validate across 3 seeds.'),
    (0.86, 'PLATEAU_PUSH',       'Score 0.86-0.87 — add programmatic solvers per-family at inference (Stage 3).'),
    (0.85, 'MARGIN_PROBE',       'Score 0.85-0.86 — run ablation (V71b DoRA) + (V71c rank_pattern).'),
    (0.84, 'BASELINE_HOLD',      'Score 0.84-0.85 — no movement. Reinvest in CoT quality (free-teacher distill).'),
    (0.0, 'ROLLBACK_V70',        'Score < 0.84 — ROLLBACK to V70. Do not submit further until corpus audit.'),
]

label, plan = 'UNDETERMINED', 'check Kaggle LB manually'
for threshold, lbl, p in DECISION_TREE:
    if score >= threshold:
        label, plan = lbl, p
        break

out = {
    'local_score': score,
    'decision_label': label,
    'next_action': plan,
    'regras_imutaveis': [
        'Reservar 2/5 slots para rollback',
        'Nunca strip_lm_head (V18)',
        'Nunca mudar >1 variavel por iteracao',
        'Sempre gate submission + kaggle_like_gate antes de submit',
    ],
}
with open(Path(CFG.output_dir) / 'decision.json', 'w') as f:
    json.dump(out, f, indent=2)
print(json.dumps(out, indent=2))
